## Phần 8: Tập kiểm tra bổ sung gồm văn bản do người viết lại (simulated)

**Trạng thái: đã chạy xong.** Kết quả ở ô cuối notebook. Phần 8 chạy trên Colab (dựng pool, chấm v1) và Kaggle (BM25, Dense).

**Lý do:** pool Test có 300 parent nhưng chỉ 21 parent là `simulated` (người viết lại). Phần lớn parent còn lại là obfuscation do máy sinh, nên chưa đủ mẫu để kết luận v1 xử lý văn bản người viết lại ra sao (v1 0,952 so với BM25 1,000 ở nhóm này, chênh đúng một parent). Phần này dựng thêm một tập chỉ gồm parent `simulated`, lớn hơn nhiều, lấy từ PAN-PC-11 (góp ý phản biện vòng 3, mục M3/B1.3).

**Câu hỏi nghiên cứu:** trên văn bản người viết lại, reranker v1 đã khoá có tốt hơn BM25 không?

**Giữ nguyên so với Test:** kho 2.000 tài liệu (nguồn đúng cộng tài liệu gây nhiễu chọn theo thứ tự băm), mô hình v1 (`model_final`), cấu hình khoá K = 50, MaxP 4 đoạn, β = 1, cách cắt câu hỏi (cửa sổ 256 từ, bước nhảy 128). **Không huấn luyện lại, không chỉnh tham số.**

**Chọn parent:**
- Chỉ lấy parent `simulated`.
- Tối đa **3 parent mỗi tài liệu nghi vấn**, chọn theo thứ tự băm cố định theo `seed` 20261125, vì các parent cùng tài liệu nghi vấn không độc lập. Corpus có 4.608 trường hợp simulated từ 105 tài liệu nghi vấn; sau khi loại trùng với 5 tập trước còn 574 trường hợp từ 66 tài liệu nghi vấn (nhiều nhất 54 trường hợp từ một tài liệu).
- Loại mọi parent, tài liệu nghi vấn và tài liệu nguồn đã xuất hiện ở Train, Validation, Test, DEV0, DEV1, DEV-v2 (bằng `--exclude-manifest`).
- Mức giới hạn 3 được chọn từ bảng đếm ở Bước 3a, trước khi chạy v1.

**Pool dựng xong:** 179 parent (từ 66 tài liệu nghi vấn), 280 truy vấn, 2.000 tài liệu (161 nguồn dương, 1.839 distractor), 399.148 chunk.

**Các bước:**
1. Cài đặt, lấy code từ GitHub (nhánh `p1-simulated-extra`).
2. Giải nén PAN-PC-11, dựng lại 3 tập DEV, giải nén pool Train và Test từ Drive.
3. Chọn giới hạn parent mỗi tài liệu nghi vấn (3a, 3b).
4. Dựng pool simulated, kiểm tra (không trùng, phân bố), duyệt governance, nén và tính SHA-256 (4a đến 4e).
5. Chạy BM25 và Dense trên Kaggle (2 notebook riêng).
6. Dựng ứng viên K = 50 và chấm bằng v1 và mô hình gốc (6a đến 6d).
7. Bảng kết quả, bootstrap theo parent và theo tài liệu nghi vấn (7a, 7b).

### Bước 1 — Cài đặt và lấy code từ GitHub

Clone nhánh `p1-simulated-extra`: là code đã khóa (v1) cộng hai tùy chọn mới ở bước dựng pool (`--annotation-type`, `--max-annotations-per-suspicious-document`). Chỉ hai file `converter.py` và `cli.py` khác bản khóa, mọi thứ khác giữ nguyên.

Ô này cũng kiểm tra các file cần dùng trên Drive (PAN-PC-11, pool Train và Test cũ, mô hình v1) và in RAM, GPU, đĩa của phiên Colab. Kết quả của Phần 8 sẽ lưu ở thư mục `simulated_extra` trên Drive để không mất khi hết phiên.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, psutil, torch
DRIVE_DIR = '/content/drive/MyDrive/KLTN'
SIM_DIR   = f'{DRIVE_DIR}/simulated_extra'        # kết quả Phần 8 lưu ở đây
os.makedirs(SIM_DIR, exist_ok=True)

BRANCH   = 'p1-simulated-extra'
REPO_URL = 'https://github.com/ngwine/evidence-grounded-report-assessment.git'
if not os.path.exists('/content/repo/.git'):
    !git clone -q --branch {BRANCH} --depth 1 {REPO_URL} /content/repo
%cd /content/repo
!git log -1 --oneline

!pip install -q -r research/requirements-research.txt
!apt-get -qq install -y unrar > /dev/null 2>&1 || apt-get -qq install -y unrar-free > /dev/null 2>&1

# các file cần dùng ở các bước sau
need = ['pan-plagiarism-corpus-2011.part1.rar', 'pan-plagiarism-corpus-2011.part2.rar',
        'pan-pc11-train-v1.zip', 'pan-pc11-test-v1.zip', 'finetune']
for name in need:
    print(('có   ' if os.path.exists(f'{DRIVE_DIR}/{name}') else 'THIẾU'), name)

print('RAM tổng: %.1f GB | còn trống: %.1f GB' % (psutil.virtual_memory().total/1e9, psutil.virtual_memory().available/1e9))
print('Đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free/1e9))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có (Runtime -> Change runtime type)')

Mounted at /content/drive
/content/repo
acc4949 (grafted, HEAD -> p1-simulated-extra, origin/p1-simulated-extra) Sửa converter.py, cli.py: thêm lọc theo loại obfuscation và giới hạn số parent mỗi tài liệu nghi vấn (cho tập simulated bổ sung)
có    pan-plagiarism-corpus-2011.part1.rar
có    pan-plagiarism-corpus-2011.part2.rar
có    pan-pc11-train-v1.zip
có    pan-pc11-test-v1.zip
có    finetune
RAM tổng: 13.6 GB | còn trống: 12.1 GB
Đĩa trống: 115 GB
GPU: không có (Runtime -> Change runtime type)


### Bước 2 — Giải nén PAN-PC-11, dựng lại 3 tập DEV, nạp pool Train và Test

Để loại trừ đúng 5 tập trước, converter cần `dataset-manifest.json` (kèm `documents`, `queries`) của từng tập. Có hai nguồn:
- **DEV0, DEV1, DEV-v2:** dựng lại bằng đúng lệnh và seed như Phần 1, rồi kiểm hash `queries.jsonl` của DEV0 so với giá trị đã ghi.
- **Train và Test:** giải nén từ Drive (`pan-pc11-train-v1.zip`, `pan-pc11-test-v1.zip`), là pool chính thức đã dùng, không dựng lại.



#### Bước 2a — Giải nén corpus và dựng DEV

In [ ]:
import hashlib, pathlib
for name in ['pan-plagiarism-corpus-2011.part1.rar', 'pan-plagiarism-corpus-2011.part2.rar']:
    if not os.path.exists(f'/content/{name}'):
        print('Đang copy', name); shutil.copy(f'{DRIVE_DIR}/{name}', f'/content/{name}')
if not os.path.exists('/content/data/extracted/pan-plagiarism-corpus-2011'):
    !unrar x -y "/content/pan-plagiarism-corpus-2011.part1.rar" /content/data/extracted/ > /dev/null
INPUT_ROOT = '/content/data/extracted/pan-plagiarism-corpus-2011/external-detection-corpus'
ARCH1, ARCH2 = '/content/pan-plagiarism-corpus-2011.part1.rar', '/content/pan-plagiarism-corpus-2011.part2.rar'
D = '/content/data/derived'
print('INPUT_ROOT tồn tại:', os.path.exists(INPUT_ROOT))

!python -m research.pan_pilot --dataset pan-pc-11 --input-root "{INPUT_ROOT}" --output-dir {D}/pan-pc11-retask-v2 \
  --dataset-version zenodo-3250095-full --split-name retask-dev-v2 --max-positive-annotations 60 \
  --max-candidate-documents 400 --max-chunk-chars 1500 --seed 20261112 --source-archive {ARCH1} --source-archive {ARCH2}

!python -m research.pan_pilot --dataset pan-pc-11 --input-root "{INPUT_ROOT}" --output-dir {D}/pan-pc11-dev0 \
  --dataset-version zenodo-3250095-v1-external-derived-pilot --split-name pan-pc-11-pilot-v1 \
  --source-archive {ARCH1} --source-archive {ARCH2}

!python -m research.pan_pilot --dataset pan-pc-11 --input-root "{INPUT_ROOT}" --output-dir {D}/pan-pc11-dev1 \
  --dataset-version zenodo-3250095-v1-external-derived-pilot --split-name pan-pc-11-dev1-v2-clean --seed 20261012 \
  --source-archive {ARCH1} --source-archive {ARCH2} --exclude-manifest {D}/pan-pc11-dev0/dataset-manifest.json

h0 = hashlib.sha256(pathlib.Path(f'{D}/pan-pc11-dev0/queries.jsonl').read_bytes()).hexdigest()
print('DEV0 queries sha256:', h0)
assert h0 == '23588a796caa67f388aaed7f968d5b152e67325e80fcb31fe9d7d858b0c82e19', 'DEV0 dựng lại không khớp bản gốc'

Đang copy pan-plagiarism-corpus-2011.part1.rar
Đang copy pan-plagiarism-corpus-2011.part2.rar
INPUT_ROOT tồn tại: True
{"chunks": 79755, "documents": 400, "manifest": "/content/data/derived/pan-pc11-retask-v2/dataset-manifest.json", "qrels": 638, "queries": 638}
{"chunks": 49152, "documents": 200, "manifest": "/content/data/derived/pan-pc11-dev0/dataset-manifest.json", "qrels": 320, "queries": 320}
{"chunks": 47523, "documents": 200, "manifest": "/content/data/derived/pan-pc11-dev1/dataset-manifest.json", "qrels": 207, "queries": 207}
DEV0 queries sha256: 23588a796caa67f388aaed7f968d5b152e67325e80fcb31fe9d7d858b0c82e19


#### Bước 2b — Nạp pool Train và Test từ Drive, kiểm tra

In [ ]:
import zipfile, json

def load_pool(zip_name, target):
    if os.path.exists(f'{target}/dataset-manifest.json'):
        return
    tmp = f'/content/_tmp_{os.path.basename(target)}'
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(f'{DRIVE_DIR}/{zip_name}') as z:
        z.extractall(tmp)
    found = next(pathlib.Path(tmp).rglob('dataset-manifest.json')).parent
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.move(str(found), target)
    shutil.rmtree(tmp, ignore_errors=True)

load_pool('pan-pc11-train-v1.zip', f'{D}/pan-pc11-train-v1')
load_pool('pan-pc11-test-v1.zip',  f'{D}/pan-pc11-test-v1')

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

for name in ['pan-pc11-train-v1', 'pan-pc11-test-v1', 'pan-pc11-dev0', 'pan-pc11-dev1', 'pan-pc11-retask-v2']:
    qs = read_jsonl(f'{D}/{name}/queries.jsonl')
    docs = sum(1 for _ in open(f'{D}/{name}/documents.jsonl', encoding='utf-8'))
    print(f'{name:22s} | parent {len({q["parent_annotation_id"] for q in qs}):4d} | câu hỏi {len(qs):5d} | tài liệu {docs:5d}')

pan-pc11-train-v1      | parent  500 | câu hỏi  4322 | tài liệu  2500
pan-pc11-test-v1       | parent  300 | câu hỏi  2606 | tài liệu  2000
pan-pc11-dev0          | parent   30 | câu hỏi   320 | tài liệu   200
pan-pc11-dev1          | parent   30 | câu hỏi   207 | tài liệu   200
pan-pc11-retask-v2     | parent   60 | câu hỏi   638 | tài liệu   400


### Bước 3a — Thống kê số parent simulated còn lại, rồi chọn giới hạn mỗi tài liệu

Trước khi dựng pool, đếm bằng chính hàm của converter (không dựng gì, không đọc kết quả mô hình):
- Tổng số trường hợp `simulated` trong corpus.
- Bao nhiêu bị loại vì trùng parent, tài liệu nghi vấn hoặc tài liệu nguồn với 5 tập trước (Train/Validation, Test, DEV0, DEV1, DEV-v2).
- Với mỗi mức giới hạn số parent mỗi tài liệu nghi vấn (1 đến 6, không giới hạn): còn bao nhiêu parent, từ bao nhiêu tài liệu nghi vấn, tài liệu lớn nhất đóng góp bao nhiêu.

Mức giới hạn được chọn **sau khi xem bảng này và trước khi chạy v1**, vì cỡ mẫu chỉ phụ thuộc vào số lượng dữ liệu, không phụ thuộc kết quả. Tiêu chí chọn: đủ parent để khoảng tin cậy có ý nghĩa (so với 21 parent simulated của Test), nhưng không để vài tài liệu nghi vấn lớn chi phối mẫu.

In [ ]:
from pathlib import Path
import collections
from research.pan_pilot import converter as C

PROFILE = C.DATASET_PROFILES['pan-pc-11']
SEED = 20261125
index = C._CorpusIndex(Path(INPUT_ROOT))
scan = C._scan_annotations(index)
fingerprint, _ = C._input_inventory_fingerprint(index, scan)
prior = [f'{D}/{n}/dataset-manifest.json' for n in
         ['pan-pc11-train-v1', 'pan-pc11-test-v1', 'pan-pc11-dev0', 'pan-pc11-dev1', 'pan-pc11-retask-v2']]
excl = C._load_split_exclusions(prior, expected_dataset_id=PROFILE['dataset_id'], expected_input_fingerprint=fingerprint)

sim_all = [a for a in scan.annotations if (a.plagiarism_type or '').casefold() == 'simulated']
print('Trường hợp simulated trong corpus:', len(sim_all), '| tài liệu nghi vấn:', len({a.suspicious_reference for a in sim_all}))

elig, cnt = C._exclude_prior_split_annotations(sim_all, dataset_id=PROFILE['dataset_id'], input_root=index.root, exclusions=excl)
print('Bị loại vì trùng parent:', cnt['annotations_excluded_by_parent_id'],
      '| tài liệu nghi vấn:', cnt['annotations_excluded_by_suspicious_document'],
      '| tài liệu nguồn:', cnt['annotations_excluded_by_source_document'],
      '| tổng hợp cần loại:', cnt['annotations_excluded'])
print('Còn lại sau khi loại:', len(elig), '| tài liệu nghi vấn:', len({a.suspicious_reference for a in elig}))

print('\ngiới hạn | parent | tài liệu nghi vấn | nhiều nhất từ 1 tài liệu')
for cap in (1, 2, 3, 4, 5, 6, None):
    kept = elig if cap is None else C._cap_per_suspicious_document(elig, dataset_id=PROFILE['dataset_id'], cap=cap, seed=SEED)
    per = collections.Counter(a.suspicious_reference for a in kept)
    print(f'{str(cap):8s} | {len(kept):6d} | {len(per):17d} | {max(per.values()) if per else 0}')

Trường hợp simulated trong corpus: 4608 | tài liệu nghi vấn: 105
Bị loại vì trùng parent: 44 | tài liệu nghi vấn: 3385 | tài liệu nguồn: 2409 | tổng hợp cần loại: 4034
Còn lại sau khi loại: 574 | tài liệu nghi vấn: 66

giới hạn | parent | tài liệu nghi vấn | nhiều nhất từ 1 tài liệu
1        |     66 |                66 | 1
2        |    127 |                66 | 2
3        |    179 |                66 | 3
4        |    219 |                66 | 4
5        |    251 |                66 | 5
6        |    273 |                66 | 6
None     |    574 |                66 | 54


### Bước 3b — Kiểm tra tập 179 parent trước khi dựng pool (giới hạn 3 mỗi tài liệu)

Chọn **giới hạn 3 mỗi tài liệu nghi vấn → 179 parent** (từ 66 tài liệu nghi vấn). Trước khi dựng pool, kiểm tra hai điều:

1. **Độ độc lập thật:** 179 parent dùng bao nhiêu tài liệu **nguồn** khác nhau, và có tài liệu nguồn nào bị nhiều parent dùng chung không (parent dùng chung nguồn cũng không độc lập).
2. **Có giống Test về độ khó không:** số truy vấn mỗi parent và độ dài đoạn nghi vấn (số từ), so với toàn bộ Test và riêng 21 parent simulated của Test. Số truy vấn được tính bằng đúng hàm cắt của converter (cửa sổ 256 từ, bước nhảy 128); kiểm chứng bằng cách cộng cho 300 parent Test phải ra đúng 2.606 truy vấn.

Các số này chỉ mô tả dữ liệu, chưa dùng kết quả mô hình.

In [ ]:
import statistics

def span_text(a, cache={}):
    if a.suspicious_path not in cache:
        cache[a.suspicious_path] = C._read_utf8_exact(a.suspicious_path)
    return cache[a.suspicious_path][a.suspicious_start:a.suspicious_end]

def span_stats(anns):
    words, wins = [], []
    for a in anns:
        t = span_text(a)
        words.append(len(C._UNICODE_WORD_RE.findall(t)))
        wins.append(len(C._query_windows(t, max_word_tokens=256, stride_word_tokens=128)))
    return words, wins

def q(v):
    v = sorted(v); n = len(v)
    return f'min {v[0]} | p25 {v[n//4]} | trung vị {statistics.median(v):g} | p75 {v[3*n//4]} | max {v[-1]} | tổng {sum(v)}'

SEL = C._cap_per_suspicious_document(elig, dataset_id=PROFILE['dataset_id'], cap=3, seed=SEED)

# --- 1. độ độc lập ---
src_use = collections.Counter(a.source_path for a in SEL)
print(f'[Tập mới] parent {len(SEL)} | tài liệu nghi vấn {len({a.suspicious_reference for a in SEL})} | tài liệu nguồn khác nhau {len(src_use)}')
print('  số parent dùng chung một nguồn: nhiều nhất', max(src_use.values()), '| số nguồn có >= 2 parent:', sum(1 for v in src_use.values() if v >= 2))

# --- 2. so với Test ---
test_ids = {x['parent_annotation_id'] for x in read_jsonl(f'{D}/pan-pc11-test-v1/queries.jsonl')}
by_id = {C._annotation_id(PROFILE['dataset_id'], a): a for a in scan.annotations}
test_all = [by_id[i] for i in test_ids]
test_sim = [a for a in test_all if (a.plagiarism_type or '').casefold() == 'simulated']

for name, anns in (('Tập mới (simulated, 179)', SEL), ('Test (cả 300)', test_all), ('Test (chỉ simulated)', test_sim)):
    w, n = span_stats(anns)
    print(f'\n{name}: {len(anns)} parent')
    print('  số từ của đoạn nghi vấn  :', q(w))
    print('  số truy vấn mỗi parent   :', q(n))
print('\nKiểm chứng: tổng truy vấn tính cho 300 parent Test =', sum(span_stats(test_all)[1]))

[Tập mới] parent 179 | tài liệu nghi vấn 66 | tài liệu nguồn khác nhau 161
  số parent dùng chung một nguồn: nhiều nhất 3 | số nguồn có >= 2 parent: 15

Tập mới (simulated, 179): 179 parent
  số từ của đoạn nghi vấn  : min 22 | p25 47 | trung vị 82 | p75 255 | max 941 | tổng 32229
  số truy vấn mỗi parent   : min 1 | p25 1 | trung vị 1 | p75 1 | max 7 | tổng 280

Test (cả 300): 300 parent
  số từ của đoạn nghi vấn  : min 30 | p25 121 | trung vị 458 | p75 2506 | max 4790 | tổng 342232
  số truy vấn mỗi parent   : min 1 | p25 1 | trung vị 3 | p75 19 | max 37 | tổng 2606

Test (chỉ simulated): 21 parent
  số từ của đoạn nghi vấn  : min 33 | p25 71 | trung vị 147 | p75 376 | max 704 | tổng 4890
  số truy vấn mỗi parent   : min 1 | p25 1 | trung vị 1 | p75 2 | max 5 | tổng 36

Kiểm chứng: tổng truy vấn tính cho 300 parent Test = 2606


### Bước 4a — Dựng pool `pan-pc11-simulated-v1`

Dựng tập bổ sung gồm các parent **simulated** (do người viết lại), giới hạn 3 parent mỗi tài liệu nghi vấn. Cấu hình giống Test: kho 2.000 tài liệu, `--max-positive-annotations 300`, cửa sổ truy vấn 256 từ, bước nhảy 128, `--max-chunk-chars 1500`. Seed 20261125.

Loại trừ cả năm split trước (train, test, dev0, dev1, retask-v2), gồm parent, tài liệu nghi vấn và toàn bộ tập nguồn ứng viên của chúng.

In [ ]:
import glob, subprocess, os, sys

# 5 manifest của các split trước (không lấy pan-pc11-simulated-v1)
EXCL = sorted(p for p in glob.glob(f'{D}/pan-pc11-*/dataset-manifest.json') if '/pan-pc11-simulated-v1/' not in p)
for p in EXCL: print(p)
print('Số manifest loại trừ:', len(EXCL))

/content/data/derived/pan-pc11-dev0/dataset-manifest.json
/content/data/derived/pan-pc11-dev1/dataset-manifest.json
/content/data/derived/pan-pc11-retask-v2/dataset-manifest.json
/content/data/derived/pan-pc11-test-v1/dataset-manifest.json
/content/data/derived/pan-pc11-train-v1/dataset-manifest.json
Số manifest loại trừ: 5


In [ ]:
OUT = f'{D}/pan-pc11-simulated-v1'
cmd = [sys.executable, '-m', 'research.pan_pilot',
       '--dataset', 'pan-pc-11', '--input-root', INPUT_ROOT, '--output-dir', OUT,
       '--dataset-version', 'zenodo-3250095-full', '--split-name', 'simulated-v1',
       '--max-positive-annotations', '300', '--max-candidate-documents', '2000',
       '--max-chunk-chars', '1500', '--seed', '20261125',
       '--annotation-type', 'simulated', '--max-annotations-per-suspicious-document', '3',
       '--source-archive', ARCH1, '--source-archive', ARCH2]
for p in EXCL: cmd += ['--exclude-manifest', p]

r = subprocess.run(cmd, capture_output=True, text=True)
print('rc =', r.returncode)
print(r.stdout[-3000:])
print(r.stderr[-3000:])

rc = 0
{"chunks": 399148, "documents": 2000, "manifest": "/content/data/derived/pan-pc11-simulated-v1/dataset-manifest.json", "qrels": 280, "queries": 280}




### Bước 4b — Kiểm tra pool `pan-pc11-simulated-v1` vừa dựng

Pool dựng xong: 2.000 tài liệu, 399.148 chunk, 280 truy vấn, 280 qrels (khớp tổng truy vấn ở Bước 3b).

Trước khi đưa lên Kaggle chạy BM25 và Dense, kiểm tra:

1. **Cấu trúc file:** tên khóa trong manifest, documents, queries, qrels (để tách tài liệu nguồn và distractor, và ngôn ngữ).
2. **Số truy vấn mỗi parent:** bao nhiêu parent chỉ có 1 truy vấn (để báo cáo Recall@1 theo parent kèm số truy vấn).
3. **Không trùng split trước:** số parent trùng với train, test, dev0, dev1, retask-v2 phải bằng 0. Số tài liệu pool trùng cho biết distractor có dùng lại của split cũ không.

In [ ]:
import json, os, glob, collections
OUT2 = os.path.dirname('/content/data/derived/pan-pc11-simulated-v1/dataset-manifest.json')
print(sorted(os.listdir(OUT2)))
man = json.load(open(f'{OUT2}/dataset-manifest.json', encoding='utf-8'))
print('khóa manifest:', list(man.keys()))

docs   = read_jsonl(f'{OUT2}/documents.jsonl')
qs     = read_jsonl(f'{OUT2}/queries.jsonl')
qrels  = read_jsonl(f'{OUT2}/qrels.jsonl')
print('khóa documents:', list(docs[0].keys()))
print('khóa qrels    :', list(qrels[0].keys()))
print('khóa queries  :', list(qs[0].keys()))

# đếm chính
parents = collections.Counter(x['parent_annotation_id'] for x in qs)
print('\ndocuments', len(docs), '| queries', len(qs), '| parents', len(parents), '| qrels', len(qrels))
print('phân bố truy vấn/parent:', dict(sorted(collections.Counter(parents.values()).items())))
print('parent chỉ 1 truy vấn:', sum(1 for v in parents.values() if v == 1))

# độ chồng lấn với các split trước
prior = [p for p in glob.glob(f'{D}/pan-pc11-*/') if os.path.abspath(p) != os.path.abspath(OUT2)]
print('\nsplit trước tìm thấy:', [os.path.basename(p.rstrip("/")) for p in prior])
new_par = set(parents)
for p in prior:
    name = os.path.basename(p.rstrip('/'))
    qf, df = f'{p}/queries.jsonl', f'{p}/documents.jsonl'
    if not (os.path.exists(qf) and os.path.exists(df)): print(name, '(thiếu file)'); continue
    pq = {x['parent_annotation_id'] for x in read_jsonl(qf)}
    pd = {x.get('document_id') or x.get('doc_id') for x in read_jsonl(df)}
    nd = {x.get('document_id') or x.get('doc_id') for x in docs}
    print(f'{name}: parent trùng {len(new_par & pq)} | tài liệu pool trùng {len(nd & pd)}')

['chunks.jsonl', 'dataset-manifest.json', 'documents.jsonl', 'qrels.jsonl', 'queries.jsonl']
khóa manifest: ['dataset_id', 'dataset_version', 'description', 'evaluation_role', 'exclusions', 'files', 'governance', 'result_classification', 'sampling', 'schema_version', 'source', 'source_extract', 'split', 'transformation']
khóa documents: ['dataset_version', 'document_id', 'raw_sha256', 'source_reference', 'text', 'version']
khóa qrels    : ['annotated_source_span', 'declared_source_reference', 'document_id', 'parent_annotation_id', 'qrel_id', 'query_id', 'query_window_index', 'raw_end', 'raw_start', 'relevance', 'source_reference', 'target_scope']
khóa queries  : ['annotation_provenance', 'parent_annotation_id', 'query_id', 'query_role', 'query_window', 'text']

documents 2000 | queries 280 | parents 179 | qrels 280
phân bố truy vấn/parent: {1: 136, 2: 12, 3: 16, 4: 5, 5: 9, 7: 1}
parent chỉ 1 truy vấn: 136

split trước tìm thấy: ['pan-pc11-dev1', 'pan-pc11-dev0', 'pan-pc11-train-v1', '

### Bước 4c — Bảng đếm của pool để duyệt trước khi chạy BM25 và Dense

In bảng đếm: tài liệu (nguồn dương và distractor), chunk, truy vấn, qrels, số nguồn khác nhau, ngôn ngữ distractor, kèm Test-v1 để so sánh. In thêm phần `exclusions` và `sampling` trong manifest để lưu làm bằng chứng.

In [ ]:
import json, collections, re

def pool_table(path):
    docs  = read_jsonl(f'{path}/documents.jsonl')
    qrels = read_jsonl(f'{path}/qrels.jsonl')
    qs    = read_jsonl(f'{path}/queries.jsonl')
    n_chunks = sum(1 for _ in open(f'{path}/chunks.jsonl', encoding='utf-8'))
    src_ids = {r['document_id'] for r in qrels}
    dist = [d for d in docs if d['document_id'] not in src_ids]
    return dict(docs=docs, qrels=qrels, qs=qs, n_chunks=n_chunks, src_ids=src_ids, dist=dist)

NEW, TST = pool_table(OUT2), pool_table(f'{D}/pan-pc11-test-v1')
print(f'{"":28}{"Simulated-v1":>10}{"Test-v1":>10}')
for lab, f in (('tài liệu pool', lambda t: len(t['docs'])),
               ('  nguồn dương (khác nhau)', lambda t: len(t['src_ids'])),
               ('  distractor', lambda t: len(t['dist'])),
               ('chunk', lambda t: t['n_chunks']),
               ('truy vấn', lambda t: len(t['qs'])),
               ('qrels', lambda t: len(t['qrels'])),
               ('parent', lambda t: len({x['parent_annotation_id'] for x in t['qs']}))):
    print(f'{lab:28}{f(NEW):>10}{f(TST):>10}')

# ngôn ngữ: xem source_reference của vài tài liệu, rồi thử suy ra
print('\nmẫu source_reference:')
for d in NEW['dist'][:3]: print(' ', d['source_reference'])
print('mẫu nguồn dương:', [d['source_reference'] for d in NEW['docs'] if d['document_id'] in NEW['src_ids']][:2])

print('\nmanifest.exclusions =', json.dumps(man['exclusions'], ensure_ascii=False)[:1500])
print('manifest.sampling   =', json.dumps(man['sampling'],   ensure_ascii=False)[:1500])

                            Simulated-v1   Test-v1
tài liệu pool                     2000      2000
  nguồn dương (khác nhau)          161       287
  distractor                      1839      1713
chunk                           399148    375876
truy vấn                           280      2606
qrels                              280      2606
parent                             179       300

mẫu source_reference:
  source-document/part16/source-document07581.txt
  source-document/part5/source-document02335.txt
  source-document/part2/source-document00618.txt
mẫu nguồn dương: ['source-document/part15/source-document07196.txt', 'source-document/part12/source-document05785.txt']

manifest.exclusions = {"manifest_count": 5, "manifests": [{"candidate_source_documents": 200, "dataset_version": "zenodo-3250095-v1-external-derived-pilot", "documents_sha256": "5dabb8b4e588a725c27b8691277077d526a8562134e50a0dd36776281cb401ea", "manifest_sha256": "404a2148549fa32f3e546d2764ac1e82a8978c76789a2b8ca

### Bước 4d — So sánh sampling của pool simulated với Test

In `sampling` của Test-v1 để đối chiếu với `sampling` của pool simulated ở Bước 4c: cùng cách sắp thứ tự (`sha256(seed, namespace, stable_id)`), cùng kho 2.000 tài liệu. Test: 287 nguồn dương và 1.713 distractor (seed 20261119); simulated: 161 nguồn dương và 1.839 distractor (seed 20261125).

In [ ]:
tm = json.load(open(f'{D}/pan-pc11-test-v1/dataset-manifest.json', encoding='utf-8'))
print({k: v for k, v in tm['sampling'].items() if k != 'selected_annotation_ids'})

{'annotations_before_exclusion': 49620, 'annotations_eligible_after_exclusion': 43314, 'annotations_excluded': 6306, 'annotations_excluded_by_parent_id': 119, 'annotations_excluded_by_source_document': 4466, 'annotations_excluded_by_suspicious_document': 2782, 'candidate_documents_selected': 2000, 'distractor_documents_available': 10021, 'distractor_documents_selected': 1713, 'full_corpus_result': False, 'max_annotations': 300, 'max_candidate_documents': 2000, 'max_positive_annotations': 300, 'ordering': 'sha256(seed, namespace, stable_id)', 'positive_annotations_available': 43314, 'positive_annotations_selected': 300, 'positive_unit': 'plagiarism_feature', 'relevant_source_documents_available': 7384, 'relevant_source_documents_selected': 287, 'seed': 20261119, 'selected_annotation_ids_sha256': 'c80333635f07d82f2135ce77ee80bb79988ab9835f3716eaaae5057cd5bc20da', 'selected_candidate_source_paths_sha256': '488ffc5db36886475832f7c7067f9ccf4c97120276823f0ad65ef2f8cc4924dc', 'source_document

### Bước 4e — Nén pool và lưu lên Drive

Nén pool `pan-pc11-simulated-v1` thành một file zip để đưa lên Kaggle, lưu bản sao vào thư mục `KLTN/simulated_extra/` trên Drive, và tính SHA-256 của file zip cùng các file chính.


In [ ]:
import shutil, hashlib, os
DRV = '/content/drive/MyDrive/KLTN/simulated_extra'
os.makedirs(DRV, exist_ok=True)

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

zip_path = shutil.make_archive(f'{DRV}/pan-pc11-simulated-v1', 'zip',
                               root_dir=os.path.dirname(OUT2), base_dir=os.path.basename(OUT2))
print('zip:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB')
print('SHA-256 zip:', sha(zip_path))
for fn in ('dataset-manifest.json', 'documents.jsonl', 'queries.jsonl', 'qrels.jsonl', 'chunks.jsonl'):
    print(f'{fn:24}', sha(f'{OUT2}/{fn}'))

zip: /content/drive/MyDrive/KLTN/simulated_extra/pan-pc11-simulated-v1.zip 438.3 MB
SHA-256 zip: 72aba8ae9c17b2ddf0ba92d33979c943c797b0b4650d9eab0973b24b6e331bd1
dataset-manifest.json    699cf58ddfa9b39ad778555e0e5ad38a7ca065710cdefc3737706ccaf5199d4b
documents.jsonl          7b1a7851d72f9187e2ac1f85f51571f7853019519bf18398252a338b30a28421
queries.jsonl            6e75f14847156a503edefefb05aa827f7b0a4c301ce44b1bb2f9bbb05e1c597e
qrels.jsonl              ad179b121de7d59d8a25764374d1f81dcd6a0718af8af53485a113bbbfa1bf24
chunks.jsonl             a50592dcfe433f420501e282b7382d9d8812414f4fd88f6e97e5c8ea4948da7b


### Bước 5 — Chạy BM25 và Dense trên Kaggle

Hai notebook riêng, chạy trên Kaggle với cùng pool `pan-pc11-simulated-v1` (`simulated_extra/pan-pc11-simulated-v1.zip`):
- BM25: CPU, 4,1 phút, ra `bm25_simulated_results.zip`.
- Dense (E5): GPU T4 fp16, 58,3 phút, ra `dense_simulated_results.zip`.

Tải hai file zip về và lưu vào Drive `KLTN/simulated_extra/` để Bước 6b đọc.

### Bước 6a — Nạp hàm chấm điểm reranker

Nạp lại đúng các hàm đã dùng khi chấm Test (`encode_pair`, `collate`, `score_rows`): cùng tokenizer, cùng độ dài tối đa (512 token, truy vấn tối đa 160), cùng độ chính xác fp16. Mô hình gốc (zero-shot) là `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`; mô hình v1 là `finetune/model_final` trên Drive (đã đóng băng, chỉ đọc, không train lại).

Kết quả đúng: in `Đã nạp hàm`.

In [ ]:
import os, json, time, zipfile, shutil, gc, pickle
import numpy as np, torch
from collections import Counter, defaultdict
from google.colab import drive
drive.mount('/content/drive')

print("GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForSequenceClassification

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
FT        = f'{DRIVE_DIR}/finetune'
SIM_DRIVE = f'{DRIVE_DIR}/simulated_extra'
MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
MAX_LEN, Q_MAX = 512, 160
tok = AutoTokenizer.from_pretrained(MODEL)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8").read().split("\n") if l.strip()]

def encode_pair(query, passage):
    q = tok(query, add_special_tokens=False, truncation=True, max_length=Q_MAX)["input_ids"]
    room = MAX_LEN - len(q) - 4
    p = tok(passage, add_special_tokens=False, truncation=True, max_length=room)["input_ids"]
    return [tok.cls_token_id] + q + [tok.sep_token_id, tok.sep_token_id] + p + [tok.sep_token_id]

_a, _b = "hello world this is a question", "an answer passage with some words"
assert encode_pair(_a, _b) == tok(_a, _b)["input_ids"], "định dạng token không khớp, báo mình"

def collate(ids_list):
    n = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), n), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), n), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
    return ids.cuda(), mask.cuda()

def logits_of(model, ids, mask):
    out = model(input_ids=ids, attention_mask=mask).logits
    return out.squeeze(-1) if out.shape[-1] == 1 else out[:, -1]

@torch.no_grad()
def score_rows(model, rows, bs=64):
    model.eval()
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["passage"]))
    scores = np.zeros(len(rows), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, mask = collate([encode_pair(rows[i]["query"], rows[i]["passage"]) for i in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = logits_of(model, ids, mask)
        scores[idx] = lg.float().cpu().numpy()
    return scores

print("Đã nạp hàm | v1 có tại:", os.path.isdir(f'{FT}/model_final'))

Mounted at /content/drive
GPU: Tesla T4


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Đã nạp hàm | v1 có tại: True


### Bước 6b — Nạp pool và kết quả BM25, Dense; dựng 50 ứng viên mỗi truy vấn

Làm ba việc:

1. Giải nén pool `pan-pc11-simulated-v1` và hai file kết quả Kaggle (BM25, Dense) từ Drive `simulated_extra/`.
2. Dựng danh sách ứng viên K = 50 theo cách đã khóa: xen kẽ top của `fixed_window_bm25` và `fixed_window_dense`, bỏ trùng. Với mỗi (truy vấn, tài liệu ứng viên), lấy tối đa 4 đoạn (một đoạn từ mỗi hệ: Dense cửa sổ, BM25 cửa sổ, Dense structure, BM25 structure) để reranker chấm ở bước sau.
3. **Tự kiểm tra:** tính lại Recall@1 mức parent của BM25 và Dense từ file predictions. Phải khớp số đã thấy ở Kaggle (BM25 cửa sổ cố định 0,933, Dense cửa sổ cố định 0,862). Lệch là có lỗi đọc dữ liệu, dừng lại, chưa chấm.

Thêm: in **trần** của danh sách K = 50 (tỷ lệ tài liệu nguồn đúng nằm trong 50 ứng viên). Reranker chỉ sắp xếp lại 50 tài liệu này nên không thể vượt trần.

Với tập này nhãn loại luôn là simulated; mã tài liệu nghi vấn lấy từ `annotation_provenance.xml_path` (một file XML = một tài liệu nghi vấn), dùng sau cho bootstrap theo tài liệu nghi vấn.

In [ ]:
POOL_ZIP  = f'{SIM_DRIVE}/pan-pc11-simulated-v1.zip'
BM25_ZIP  = f'{SIM_DRIVE}/bm25_simulated_results.zip'
DENSE_ZIP = f'{SIM_DRIVE}/dense_simulated_results.zip'
SIM_DIR   = '/content/data/derived/pan-pc11-simulated-v1'
K_LOCK, MAX_CHARS = 50, 1500
for f in (POOL_ZIP, BM25_ZIP, DENSE_ZIP): assert os.path.exists(f), f"Thiếu {f} trên Drive"

need = ('documents.jsonl', 'queries.jsonl', 'qrels.jsonl')
if not all(os.path.exists(f'{SIM_DIR}/{n}') for n in need):
    os.makedirs(SIM_DIR, exist_ok=True)
    with zipfile.ZipFile(POOL_ZIP) as z:
        names = z.namelist()
        for want in need:
            hits = [n for n in names if n == want or n.endswith('/' + want)]
            assert hits, f"Không thấy {want} trong zip. Vài tên đầu: {names[:8]}"
            with z.open(hits[0]) as src, open(f'{SIM_DIR}/{want}', 'wb') as dst:
                shutil.copyfileobj(src, dst)

def load_preds(zip_path, suffix, method):
    out = {}
    with zipfile.ZipFile(zip_path) as z:
        hits = [n for n in z.namelist() if n.endswith(suffix)]
        assert len(hits) == 1, f"{suffix}: tìm thấy {hits}"
        with z.open(hits[0]) as f:
            for raw in f:
                if not raw.strip(): continue
                r = json.loads(raw)
                if r["method"] == method: out[r["query_id"]] = r["candidates"]
    return out

def union_box(a, b, K):
    out, seen, i = [], set(), 0
    while len(out) < K and (i < len(a) or i < len(b)):
        for lst in (a, b):
            if i < len(lst) and lst[i]["document_id"] not in seen and len(out) < K:
                seen.add(lst[i]["document_id"]); out.append(lst[i]["document_id"])
        i += 1
    return out

docs      = {d["document_id"]: d["text"] for d in read_jsonl(f'{SIM_DIR}/documents.jsonl')}
q_rows    = read_jsonl(f'{SIM_DIR}/queries.jsonl')
queries   = {q["query_id"]: q["text"] for q in q_rows}
q2p       = {q["query_id"]: q["parent_annotation_id"] for q in q_rows}
susp_of   = {}                                        # parent -> tài liệu nghi vấn (xml_path)
for q in q_rows: susp_of.setdefault(q["parent_annotation_id"], q["annotation_provenance"]["xml_path"])
gold = defaultdict(set)
for r in read_jsonl(f'{SIM_DIR}/qrels.jsonl'): gold[r["query_id"]].add(r["document_id"])
assert set(queries) == set(gold) == set(q2p), "câu hỏi/qrels/queries không khớp nhau"
print("Pool: tài liệu", len(docs), "| câu hỏi", len(queries), "| parent", len(set(q2p.values())),
      "| tài liệu nghi vấn", len(set(susp_of.values())))

M = {"bm": ("bm25", "fixed_window_bm25"), "de": ("dense", "fixed_window_dense"),
     "sbm": ("bm25", "structure_bm25"), "sde": ("dense", "structure_dense")}
P = {k: load_preds(BM25_ZIP if z == "bm25" else DENSE_ZIP, f"{z}_chunking/predictions.jsonl", m)
     for k, (z, m) in M.items()}
for k in P: assert set(P[k]) == set(queries), f"{k}: thiếu câu hỏi trong predictions"

span_of = {}
for k in M:
    for q, cl in P[k].items():
        for c in cl:
            if c["source_spans"]:
                s0 = c["source_spans"][0]; span_of[(k, q, c["document_id"])] = (s0["raw_start"], s0["raw_end"])

box = {q: union_box(P["bm"][q], P["de"][q], K_LOCK) for q in queries}
rrf_sc, rrf_order = {}, {}
for q in queries:
    sc = defaultdict(float)
    for k in ("bm", "de"):
        for c in P[k][q]: sc[c["document_id"]] += 1.0 / (60 + c["rank"])
    rrf_sc[q] = sc; rrf_order[q] = sorted(sc, key=lambda d: (-sc[d], d))[:100]

rows_s, meta_s = [], []
for q in queries:
    for d in box[q]:
        seen = set()
        for kind in ("de", "bm", "sde", "sbm"):
            sp = span_of.get((kind, q, d))
            if sp and sp not in seen:
                seen.add(sp); text = docs[d][sp[0]:sp[1]][:MAX_CHARS]
                if text.strip():
                    rows_s.append({"query": queries[q], "passage": text}); meta_s.append((q, d, kind))
print(f"\nSố cặp cần chấm: {len(rows_s)} | trung bình {len(rows_s)/(len(queries)*K_LOCK):.2f} đoạn mỗi ứng viên")
print(f"Ước lượng mỗi lần chấm: {len(rows_s)/154/60:.0f} phút (theo 154 cặp/giây)")

KS = (1, 10, 100)
def parent_recall(ranked):
    acc = defaultdict(lambda: {k: [] for k in KS})
    for q, lst in ranked.items():
        g = gold[q]
        for k in KS: acc[q2p[q]][k].append(len(set(lst[:k]) & g) / len(g))
    return {p: {k: float(np.mean(v[k])) for k in KS} for p, v in acc.items()}
for name, rk in (("BM25 (cửa sổ)", {q: [c["document_id"] for c in P["bm"][q]] for q in queries}),
                 ("Dense (cửa sổ)", {q: [c["document_id"] for c in P["de"][q]] for q in queries}),
                 ("RRF", rrf_order)):
    pm = parent_recall(rk)
    print(f"  {name:15s}: R@1 {np.mean([v[1] for v in pm.values()]):.3f} | R@10 {np.mean([v[10] for v in pm.values()]):.3f} | R@100 {np.mean([v[100] for v in pm.values()]):.3f}")
print("  (phải khớp Kaggle: BM25 R@1 = 0.933, Dense R@1 = 0.862)")

pc = defaultdict(list)
for q in queries: pc[q2p[q]].append(len(gold[q] & set(box[q])) / len(gold[q]))
print(f"Trần của danh sách xen kẽ K = {K_LOCK}: {np.mean([np.mean(v) for v in pc.values()]):.3f}")

Pool: tài liệu 2000 | câu hỏi 280 | parent 179 | tài liệu nghi vấn 66

Số cặp cần chấm: 35233 | trung bình 2.52 đoạn mỗi ứng viên
Ước lượng mỗi lần chấm: 4 phút (theo 154 cặp/giây)
  BM25 (cửa sổ)  : R@1 0.933 | R@10 0.968 | R@100 1.000
  Dense (cửa sổ) : R@1 0.862 | R@10 0.933 | R@100 0.978
  RRF            : R@1 0.916 | R@10 0.978 | R@100 0.994
  (phải khớp Kaggle: BM25 R@1 = 0.933, Dense R@1 = 0.862)
Trần của danh sách xen kẽ K = 50: 0.994


### Bước 6c — Chấm điểm 35.233 cặp bằng v1 (đã đóng băng)

Chấm từng cặp (truy vấn, đoạn) bằng reranker v1 (`finetune/model_final`, chỉ đọc, không train lại). Điểm của tài liệu là điểm cao nhất trong các đoạn của nó (MaxP, tối đa 4 đoạn mỗi tài liệu). Điểm được lưu từng phần xuống Drive `simulated_extra/` nên nếu phiên bị ngắt thì chạy lại sẽ bỏ qua phần đã xong.

Kết quả đúng: in `Xong v1: ... cặp (câu hỏi, tài liệu)`.

In [ ]:
CHUNK = 30000
def score_ckpt_sim(model, tag):
    path = lambda i: f'{SIM_DRIVE}/sim_{tag}_part{i:03d}.npy'
    parts, t0 = [], time.time()
    for i, s0 in enumerate(range(0, len(rows_s), CHUNK)):
        if os.path.exists(path(i)):
            parts.append(np.load(path(i))); continue
        sc = score_rows(model, rows_s[s0:s0 + CHUNK], bs=64)
        np.save(path(i), sc); parts.append(sc)
        print(f"  {tag}: xong {min(s0 + CHUNK, len(rows_s))}/{len(rows_s)} cặp | {(time.time() - t0) / 60:.1f} phút", flush=True)
    scores = np.concatenate(parts)
    assert len(scores) == len(rows_s), f"điểm đã lưu ({len(scores)}) lệch số cặp hiện tại ({len(rows_s)}): xoá file sim_{tag}_part*.npy"
    d = {}
    for (q, doc, kind), v in zip(meta_s, scores):
        d[(q, doc)] = max(float(v), d.get((q, doc), -1e9))
    pickle.dump(d, open(f'{SIM_DRIVE}/sim_scores_{tag}.pkl', 'wb'))
    return d

model_v1 = AutoModelForSequenceClassification.from_pretrained(f'{FT}/model_final').cuda()
sim_v1 = score_ckpt_sim(model_v1, "finetuned_v1")
print("Xong v1:", len(sim_v1), "cặp (câu hỏi, tài liệu)")
del model_v1; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  finetuned_v1: xong 30000/35233 cặp | 2.5 phút
  finetuned_v1: xong 35233/35233 cặp | 3.0 phút
Xong v1: 14000 cặp (câu hỏi, tài liệu)


### Bước 6d — Chấm điểm bằng mô hình gốc (zero-shot)

Cùng 35.233 cặp, cùng cách chấm, nhưng dùng mô hình gốc chưa fine-tune (`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`). Dùng làm mốc "trước khi fine-tune" để thấy fine-tune đóng góp bao nhiêu. Phải chạy sau Bước 6c trong cùng phiên (cần hàm `score_ckpt_sim` và danh sách cặp).

Kết quả đúng: in `Xong mô hình gốc: ... cặp (câu hỏi, tài liệu)`.

In [ ]:
model_zs = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
sim_zs = score_ckpt_sim(model_zs, "zeroshot")
print("Xong mô hình gốc:", len(sim_zs), "cặp (câu hỏi, tài liệu)")
del model_zs; torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  zeroshot: xong 30000/35233 cặp | 2.6 phút
  zeroshot: xong 35233/35233 cặp | 3.0 phút
Xong mô hình gốc: 14000 cặp (câu hỏi, tài liệu)


### Bước 7a — Bảng kết quả: BM25, Dense, RRF, mô hình gốc, v1

Xếp hạng cuối của reranker: trong 50 ứng viên, sắp theo điểm reranker (β = 1, đã khóa ở Bước 16 của notebook `p1_colab_build_pools_and_finetune` trên tập val); các tài liệu ngoài 50 ứng viên xếp sau theo RRF. Dùng cùng hàm `final_ranking` như lúc chấm Test.

Bảng báo cáo Recall mức **parent** (179 parent, mỗi parent một lần), kèm Recall@1 mức truy vấn (280 truy vấn) để thấy số truy vấn mỗi parent không làm lệch. Không gộp nhiều truy vấn thành một điểm cho parent.

Điểm lưu xuống Drive `simulated_extra/` để dùng lại.

In [ ]:
BETA = 1.0   # khóa ở Bước 16 (β = 1, chỉ điểm reranker)
sc = {"gốc (zero-shot)": sim_zs, "v1 (fine-tune)": sim_v1}

def zs(x):
    x = np.asarray(x, float); sd = x.std()
    return (x - x.mean()) / sd if sd > 0 else x * 0

def final_ranking(s, beta=BETA, K=K_LOCK):
    ranked = {}
    for q in queries:
        b = box[q][:K]; inbox = set(b)
        have = [d for d in b if (q, d) in s]; miss = [d for d in b if (q, d) not in s]
        if have:
            fin = beta * zs([s[(q, d)] for d in have]) + (1 - beta) * zs([rrf_sc[q].get(d, 0.0) for d in have])
            have = [d for _, d in sorted(zip(-fin, have), key=lambda t: (t[0], t[1]))]
        ranked[q] = (have + miss + [d for d in rrf_order[q] if d not in inbox])[:100]
    return ranked

results = {"BM25": {q: [c["document_id"] for c in P["bm"][q]] for q in queries},
           "Dense": {q: [c["document_id"] for c in P["de"][q]] for q in queries},
           "RRF": rrf_order}
for n in sc: results[n] = final_ranking(sc[n])
PM = {n: parent_recall(rk) for n, rk in results.items()}

def query_r1(ranked):
    return float(np.mean([len(set(ranked[q][:1]) & gold[q]) / len(gold[q]) for q in queries]))

print(f"{'phương pháp':18s} | {'parent R@1':>10s} | {'R@10':>6s} | {'R@100':>6s} | {'truy vấn R@1':>12s}   (179 parent, 280 truy vấn)")
for n, pm in PM.items():
    print(f"{n:18s} | " + " | ".join(f"{np.mean([pm[p][k] for p in pm]):{w}.3f}" for k, w in ((1, 10), (10, 6), (100, 6)))
          + f" | {query_r1(results[n]):12.3f}")

pickle.dump({"PM": PM, "results": results}, open(f'{SIM_DRIVE}/sim_final_rankings.pkl', 'wb'))
print("\nĐã lưu sim_final_rankings.pkl")

phương pháp        | parent R@1 |   R@10 |  R@100 | truy vấn R@1   (179 parent, 280 truy vấn)
BM25               |      0.933 |  0.968 |  1.000 |        0.950
Dense              |      0.862 |  0.933 |  0.978 |        0.907
RRF                |      0.916 |  0.978 |  0.994 |        0.939
gốc (zero-shot)    |      0.795 |  0.937 |  0.994 |        0.761
v1 (fine-tune)     |      0.966 |  0.989 |  0.994 |        0.971

Đã lưu sim_final_rankings.pkl


### Bước 7b — Bootstrap: v1 có hơn các hệ khác không?

Tính hiệu số **v1 − hệ khác** trên từng parent (Recall@1 và Recall@10), rồi ước lượng khoảng tin cậy 95% bằng bootstrap 2.000 lần (seed 20261121, giống các bước trước). Làm hai cách:

1. **Theo parent** (179 đơn vị): bốc ngẫu nhiên có hoàn lại các parent. Đây là cách báo cáo chính, giống Test.
2. **Theo tài liệu nghi vấn** (66 đơn vị): bốc có hoàn lại các tài liệu nghi vấn, lấy kèm toàn bộ parent của chúng. Đây là phân tích độ nhạy, vì các parent cùng một tài liệu nghi vấn không hoàn toàn độc lập (một tài liệu có tới 3 parent).

Dấu `*` nghĩa là khoảng 95% không chứa 0. Cột `thắng/thua/hòa` đếm số parent v1 tốt hơn, kém hơn, bằng hệ kia ở Recall@1, để thấy mức chênh lệch đến từ bao nhiêu parent (rất nhỏ ở tập này).

Kết quả: hai cách cho khoảng tin cậy gần như trùng nhau (ví dụ v1 − BM25 ở R@1 đều là [+0,006, +0,067]), nên việc các parent cùng tài liệu nghi vấn không độc lập hầu như không đổi kết luận.

In [ ]:
rng = np.random.default_rng(20261121)
parents = sorted(PM["BM25"])
susp_groups = defaultdict(list)
for p in parents: susp_groups[susp_of[p]].append(p)
susp_list = sorted(susp_groups)

def boot_parent(a, b, k, n=2000):
    d = np.array([PM[a][p][k] - PM[b][p][k] for p in parents])
    idx = rng.integers(0, len(parents), (n, len(parents)))
    return d.mean(), np.percentile(d[idx].mean(1), [2.5, 97.5])

def boot_susp(a, b, k, n=2000):
    s = np.array([sum(PM[a][p][k] - PM[b][p][k] for p in susp_groups[g]) for g in susp_list])
    c = np.array([len(susp_groups[g]) for g in susp_list], float)
    idx = rng.integers(0, len(susp_list), (n, len(susp_list)))
    return s.sum() / c.sum(), np.percentile(s[idx].sum(1) / c[idx].sum(1), [2.5, 97.5])

A = "v1 (fine-tune)"
print(f"{len(parents)} parent | {len(susp_list)} tài liệu nghi vấn | parent mỗi tài liệu nghi vấn: {dict(sorted(Counter(len(v) for v in susp_groups.values()).items()))}\n")
for b in ("BM25", "Dense", "RRF", "gốc (zero-shot)"):
    for k in (1, 10):
        mp, cp = boot_parent(A, b, k); ms, cs = boot_susp(A, b, k)
        sp = "*" if (cp[0] > 0 or cp[1] < 0) else " "; ss = "*" if (cs[0] > 0 or cs[1] < 0) else " "
        d1 = [PM[A][p][1] - PM[b][p][1] for p in parents] if k == 1 else None
        wl = f" | thắng/thua/hòa {sum(x > 0 for x in d1)}/{sum(x < 0 for x in d1)}/{sum(x == 0 for x in d1)}" if d1 else ""
        print(f"v1 − {b:16s} R@{k:<2d} | theo parent {mp:+.3f} [{cp[0]:+.3f}, {cp[1]:+.3f}] {sp} | theo tài liệu nghi vấn {ms:+.3f} [{cs[0]:+.3f}, {cs[1]:+.3f}] {ss}{wl}")

179 parent | 66 tài liệu nghi vấn | parent mỗi tài liệu nghi vấn: {1: 5, 2: 9, 3: 52}

v1 − BM25             R@1  | theo parent +0.034 [+0.006, +0.067] * | theo tài liệu nghi vấn +0.034 [+0.006, +0.067] * | thắng/thua/hòa 7/1/171
v1 − BM25             R@10 | theo parent +0.020 [+0.004, +0.045] * | theo tài liệu nghi vấn +0.020 [+0.004, +0.042] *
v1 − Dense            R@1  | theo parent +0.104 [+0.058, +0.158] * | theo tài liệu nghi vấn +0.104 [+0.057, +0.153] * | thắng/thua/hòa 22/3/154
v1 − Dense            R@10 | theo parent +0.056 [+0.022, +0.095] * | theo tài liệu nghi vấn +0.056 [+0.023, +0.094] *
v1 − RRF              R@1  | theo parent +0.050 [+0.017, +0.089] * | theo tài liệu nghi vấn +0.050 [+0.017, +0.086] * | thắng/thua/hòa 10/1/168
v1 − RRF              R@10 | theo parent +0.011 [-0.011, +0.034]   | theo tài liệu nghi vấn +0.011 [-0.011, +0.033]  
v1 − gốc (zero-shot)  R@1  | theo parent +0.172 [+0.122, +0.220] * | theo tài liệu nghi vấn +0.172 [+0.119, +0.225] * | thắng/th

### Kết quả Phần 8: v1 so với BM25 trên 179 parent simulated

Pool: 179 parent từ 66 tài liệu nghi vấn, 280 truy vấn, 2.000 tài liệu (161 nguồn dương, 1.839 distractor), 399.148 chunk. Chấm 35.233 cặp mất 3,0 phút trên T4 cho mỗi mô hình.

| Hệ | R@1 | R@10 | R@100 | v1 − hệ, R@1 [95% CI theo parent] | thắng/thua/hòa |
|---|---:|---:|---:|---|---|
| BM25 | 0,933 | 0,968 | 1,000 | +0,034 [+0,006, +0,067] | 7/1/171 |
| Dense | 0,862 | 0,933 | 0,978 | +0,104 [+0,058, +0,158] | 22/3/154 |
| RRF | 0,916 | 0,978 | 0,994 | +0,050 [+0,017, +0,089] | 10/1/168 |
| Mô hình gốc (zero-shot) | 0,795 | 0,937 | 0,994 | +0,172 [+0,122, +0,220] | 47/1/131 |
| v1 | 0,966 | 0,989 | 0,994 | – | – |

**Cách đọc:**
- Mức parent (179) là số báo cáo chính. Ở R@1 mức truy vấn, v1 đạt 0,971 và BM25 đạt 0,950.
- v1 hơn BM25 ở R@1 (+0,034, khoảng tin cậy không chứa 0) nhưng biên rất mỏng: 7 parent v1 thắng, 1 parent thua, 171 hòa, và cận dưới chỉ +0,006. Ở R@10, v1 hơn BM25 +0,020 [+0,004, +0,045] và không hơn RRF (+0,011 [−0,011, +0,034], chứa 0).
- R@100 của v1 (0,994) thấp hơn BM25 (1,000) vì reranker chỉ sắp xếp lại 50 ứng viên, và danh sách này chứa nguồn đúng cho 0,994 số parent (trần K = 50).
- Bootstrap theo tài liệu nghi vấn (66 đơn vị) cho khoảng tin cậy gần như trùng với theo parent.
- Không hiệu chỉnh so sánh bội. Tập này không có parent dịch hay artificial, nên không nói được gì về hai nhóm đó.

Điểm lưu ở `simulated_extra/sim_final_rankings.pkl`. SHA-256 của pool nằm ở Bước 4e.